# Phase 5 — MC Dropout Uncertainty (Colab T4)

**Pre-requisite:** Phase 4 done — `models/robust/best.pt` in Drive, 25 corrupted sets.

## ⚠️ Important — stock RT-DETR has zero-probability dropout
RT-DETR-L's decoder has `nn.Dropout` layers but all default to `p=0.0` — a no-op.
Naive MC Dropout would give 30 identical passes and zero uncertainty. This notebook
**injects `p=0.1` into the existing decoder dropout modules** (no new modules, so the
`.pt` weights still load). If post-hoc injection yields ECE > 0.10, fall back to the
retrain cell at the end.

**Targets:** ECE <= 0.10; Spearman rho > 0.3; severity-5 uncertainty >= 1.5x severity-1
(for at least 3/5 corruptions).

In [ ]:
import os, json, shutil, random
import cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

PROJECT_DIR = '/content/drive/MyDrive/robot-perception'
RESULTS_DIR = f'{PROJECT_DIR}/results/figures'
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASS_NAMES = ['arm', 'leg', 'torso', 'head', 'sensor']
CORRUPTIONS = ['gaussian_noise', 'motion_blur', 'gaussian_blur', 'brightness', 'occlusion']
SEVERITIES  = [1, 2, 3, 4, 5]
DEVICE = 'cuda'
N_PASSES = 30

## Step 1 — MC Dropout helper functions

Inlined from `scripts/mc_dropout_inference.py` (device set to `cuda` for Colab).

In [ ]:
def enable_mc_dropout(model):
    """Keep all Dropout layers in train mode so they stay stochastic at inference."""
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()
    return model

def inject_dropout(model, p=0.1):
    """Flip existing nn.Dropout modules to p>0 (stock RT-DETR ships them at p=0.0).
    Only changes the probability attribute — no new modules, weights still load."""
    n = 0
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.p = p
            n += 1
    print(f'Set p={p} on {n} Dropout modules.')
    return model

def _box_iou(a, b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1]); x2=min(a[2],b[2]); y2=min(a[3],b[3])
    inter = max(0,x2-x1)*max(0,y2-y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

def verify_mc_dropout_active(model, img_path):
    """Run the same image twice — outputs must differ if dropout is active."""
    model.model.eval()
    enable_mc_dropout(model.model)
    r1 = model(img_path, verbose=False)[0]
    r2 = model(img_path, verbose=False)[0]
    if len(r1.boxes) == 0 or len(r2.boxes) == 0:
        print('No detections — try another image.')
        return False
    c1, c2 = float(r1.boxes.conf[0]), float(r2.boxes.conf[0])
    active = abs(c1 - c2) > 1e-6
    print(f'MC Dropout {"ACTIVE" if active else "INACTIVE"} — pass1={c1:.4f} pass2={c2:.4f}')
    return active

def mc_dropout_inference(model, img_path, n_passes=30, iou_match=0.4):
    """N stochastic passes; per reference-box confidence mean + std (uncertainty)."""
    model.model.eval()
    enable_mc_dropout(model.model)
    passes = [model(img_path, verbose=False)[0] for _ in range(n_passes)]
    ref = passes[0]
    if len(ref.boxes) == 0:
        return []
    ref_boxes = ref.boxes.xyxy.cpu().numpy()
    ref_cls   = ref.boxes.cls.cpu().numpy().astype(int)
    out = []
    for box, cls in zip(ref_boxes, ref_cls):
        confs = []
        for pr in passes:
            if len(pr.boxes) == 0:
                confs.append(0.0); continue
            cb = pr.boxes.xyxy.cpu().numpy(); cc = pr.boxes.conf.cpu().numpy()
            best_iou, best_conf = 0.0, 0.0
            for j in range(len(cb)):
                iou = _box_iou(box, cb[j])
                if iou > best_iou:
                    best_iou, best_conf = iou, float(cc[j])
            confs.append(best_conf if best_iou >= iou_match else 0.0)
        out.append({'box': box.tolist(), 'class': int(cls),
                    'mean_confidence': float(np.mean(confs)),
                    'uncertainty': float(np.std(confs))})
    return out

def compute_ECE(predictions, n_buckets=10):
    """Expected Calibration Error. Each pred: mean_confidence + is_correct."""
    n = len(predictions)
    if n == 0:
        return 0.0
    bs = 1.0 / n_buckets
    ece = 0.0
    for i in range(n_buckets):
        lo, hi = i*bs, (i+1)*bs
        bucket = [p for p in predictions if lo <= p['mean_confidence'] < hi]
        if not bucket:
            continue
        ac = np.mean([p['mean_confidence'] for p in bucket])
        aa = np.mean([float(p['is_correct']) for p in bucket])
        ece += (len(bucket)/n) * abs(ac - aa)
    return float(ece)

## Step 2 — Load model + inject dropout + verify

In [ ]:
from ultralytics import RTDETR

# Copy data + corrupted sets to local disk
LOCAL_DATA = '/content/robot_data'
if not os.path.exists(LOCAL_DATA):
    shutil.copytree(f'{PROJECT_DIR}/data/annotated', LOCAL_DATA)
CORRUPTED_LOCAL = '/content/corrupted'
if not os.path.exists(CORRUPTED_LOCAL) and os.path.exists(f'{PROJECT_DIR}/data/corrupted'):
    shutil.copytree(f'{PROJECT_DIR}/data/corrupted', CORRUPTED_LOCAL)

model = RTDETR(f'{PROJECT_DIR}/models/robust/best.pt')
inject_dropout(model.model, p=0.1)

test_imgs = sorted(f'{LOCAL_DATA}/images/test/{f}'
                   for f in os.listdir(f'{LOCAL_DATA}/images/test')
                   if f.lower().endswith(('.jpg','.jpeg','.png')))

active = verify_mc_dropout_active(model, test_imgs[0])
if not active:
    for ip in test_imgs[1:6]:
        if verify_mc_dropout_active(model, ip):
            active = True; break
assert active, 'MC Dropout not active — increase p or check decoder dropout modules.'

## Step 3 — MC Dropout on clean test set + ECE

A detection is `is_correct` if it matches a same-class ground-truth box at IoU >= 0.5.

In [ ]:
def load_gt(label_path, w, h):
    """Parse YOLO label -> list of (class, [x1,y1,x2,y2]) in pixels."""
    gt = []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                p = line.strip().split()
                if len(p) < 5:
                    continue
                cls = int(p[0])
                cx,cy,bw,bh = map(float, p[1:5])
                gt.append((cls, [(cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h]))
    return gt

import time
predictions = []
t0 = time.time()
for ip in test_imgs:
    img = cv2.imread(ip); h, w = img.shape[:2]
    stem = os.path.splitext(os.path.basename(ip))[0]
    gt = load_gt(f'{LOCAL_DATA}/labels/test/{stem}.txt', w, h)
    dets = mc_dropout_inference(model, ip, n_passes=N_PASSES)
    for d in dets:
        is_correct = any(d['class'] == gc and _box_iou(d['box'], gb) >= 0.5
                          for gc, gb in gt)
        d['is_correct'] = is_correct
        predictions.append(d)
elapsed = time.time() - t0

ece = compute_ECE(predictions)
lat = elapsed / max(1, len(test_imgs))
print(f'Detections: {len(predictions)}')
print(f'ECE: {ece:.4f}  (target <= 0.10)')
print(f'Latency: {lat:.2f} s/image at N={N_PASSES} passes')
print('✅ PASS' if ece <= 0.10 else '⚠️  ECE > 0.10 — see temperature-scaling / retrain cells below')

## VIZ 5.A — Reliability diagram

In [ ]:
def plot_reliability_diagram(preds, n_buckets=10):
    bs = 1.0 / n_buckets
    bc, ba = [], []
    for i in range(n_buckets):
        lo, hi = i*bs, (i+1)*bs
        bucket = [p for p in preds if lo <= p['mean_confidence'] < hi]
        if not bucket:
            continue
        bc.append(np.mean([p['mean_confidence'] for p in bucket]))
        ba.append(np.mean([float(p['is_correct']) for p in bucket]))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot([0,1],[0,1],'k--',label='Perfect calibration')
    axes[0].bar(bc, ba, width=bs*0.8, alpha=0.6, color='steelblue', label='Model')
    axes[0].set_xlabel('Mean predicted confidence'); axes[0].set_ylabel('Actual accuracy')
    axes[0].set_title('VIZ 5.A — Reliability diagram')
    axes[0].set_xlim(0,1); axes[0].set_ylim(0,1); axes[0].legend()
    axes[1].hist([p['mean_confidence'] for p in preds], bins=20,
                 color='darkorange', edgecolor='white')
    axes[1].set_xlabel('Predicted confidence'); axes[1].set_ylabel('Count')
    axes[1].set_title('Confidence distribution')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/viz5a_reliability_diagram.png', dpi=150)
    plt.show()

plot_reliability_diagram(predictions)
print('⚠️  Bars above diagonal = overconfident. Below = underconfident. Diagonal = good.')

## Step 4 — Uncertainty correlates with error (Spearman)

In [ ]:
unc   = [p['uncertainty'] for p in predictions]
error = [0 if p['is_correct'] else 1 for p in predictions]
rho, pval = spearmanr(unc, error)
print(f'Spearman rho (uncertainty vs error): {rho:.4f}  (target > 0.3)')
print(f'p-value: {pval:.2e}')
print('✅ PASS' if rho > 0.3 else '⚠️  FAIL — uncertainty not predictive of error')

## VIZ 5.B — Uncertainty overlay (3 clean + 3 severity-5)

In [ ]:
def overlay_uncertainty(img_path, dets):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB).copy()
    if not dets:
        return img
    us = [d['uncertainty'] for d in dets]
    umin, umax = min(us), max(us)
    for d in dets:
        u = (d['uncertainty']-umin)/(umax-umin+1e-8)
        color = (int(255*u), int(255*(1-u)), 0)  # green->red
        x1,y1,x2,y2 = map(int, d['box'])
        cv2.rectangle(img,(x1,y1),(x2,y2),color,3)
        cv2.putText(img, f"{CLASS_NAMES[d['class']]} u={d['uncertainty']:.2f}",
                    (x1,max(y1-8,0)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return img

clean_sample = random.sample(test_imgs, 3)
s5_dir = f'{CORRUPTED_LOCAL}/{CORRUPTIONS[0]}/severity_5/images'
s5_sample = random.sample(
    [f'{s5_dir}/{f}' for f in os.listdir(s5_dir) if f.endswith(('.jpg','.png','.jpeg'))], 3)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for col, ip in enumerate(clean_sample):
    vis = overlay_uncertainty(ip, mc_dropout_inference(model, ip, n_passes=N_PASSES))
    axes[0, col].imshow(vis); axes[0, col].set_title('Clean'); axes[0, col].axis('off')
for col, ip in enumerate(s5_sample):
    vis = overlay_uncertainty(ip, mc_dropout_inference(model, ip, n_passes=N_PASSES))
    axes[1, col].imshow(vis); axes[1, col].set_title('Corrupted (severity 5)'); axes[1, col].axis('off')
plt.suptitle('VIZ 5.B — Uncertainty overlay: GREEN=confident, RED=uncertain', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz5b_uncertainty_overlay.png', dpi=120)
plt.show()
print('⚠️  Row 2 (corrupted) should have visibly more red than row 1 (clean).')

## Step 5 — Uncertainty vs severity

Corrupted sets are subsampled to ~20 images per set to keep N=30 runtime tractable on T4.

In [ ]:
SUBSAMPLE = 20
unc_by_corruption = {}

for corruption in CORRUPTIONS:
    means = []
    for severity in SEVERITIES:
        img_dir = f'{CORRUPTED_LOCAL}/{corruption}/severity_{severity}/images'
        imgs = sorted(f'{img_dir}/{f}' for f in os.listdir(img_dir)
                      if f.endswith(('.jpg','.png','.jpeg')))[:SUBSAMPLE]
        all_u = []
        for ip in imgs:
            all_u += [d['uncertainty'] for d in mc_dropout_inference(model, ip, n_passes=N_PASSES)]
        means.append(float(np.mean(all_u)) if all_u else 0.0)
    unc_by_corruption[corruption] = means
    print(f'{corruption}: {[round(m,3) for m in means]}')

plt.figure(figsize=(9, 5))
for corruption, means in unc_by_corruption.items():
    plt.plot(SEVERITIES, means, marker='o', label=corruption)
plt.xlabel('Corruption Severity'); plt.ylabel('Mean Uncertainty')
plt.title('Uncertainty Increases with Corruption Severity')
plt.xticks(SEVERITIES); plt.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/uncertainty_vs_severity.png', dpi=150)
plt.show()

# S5/S1 ratio check — need >= 1.5x for at least 3/5 corruptions
ratios = {}
n_pass = 0
for corruption, means in unc_by_corruption.items():
    r = means[4] / means[0] if means[0] > 0 else 0.0
    ratios[corruption] = r
    ok = r >= 1.5
    n_pass += ok
    print(f'  {corruption}: S5/S1 = {r:.2f}x  {"✅" if ok else "⚠️"}')
print(f'\n{n_pass}/5 corruptions hit the 1.5x ratio (need >= 3).')
print('✅ PASS' if n_pass >= 3 else '⚠️  FAIL')

## Step 6 — Save metrics

In [ ]:
mc_metrics = {
    'ece': ece,
    'spearman_rho': float(rho),
    'n_passes': N_PASSES,
    'latency_s_per_image': lat,
    'severity_ratios': ratios,
    'uncertainty_by_corruption': unc_by_corruption,
}
with open(f'{PROJECT_DIR}/results/mc_dropout_metrics.json', 'w') as f:
    json.dump(mc_metrics, f, indent=2)
print('Saved results/mc_dropout_metrics.json')

## Fallback A — Temperature scaling (if 0.10 < ECE <= 0.15)

Fit a single scalar T on the val set; divide confidences by T. Cheap post-hoc fix.

In [ ]:
# Run ONLY if ECE is between 0.10 and 0.15.
# Grid-search a temperature on a held-out split, pick the T minimising ECE.
#
# best_T, best_ece = 1.0, ece
# for T in np.linspace(0.5, 3.0, 26):
#     scaled = [{'mean_confidence': min(1.0, p['mean_confidence']/T),
#                'is_correct': p['is_correct']} for p in predictions]
#     e = compute_ECE(scaled)
#     if e < best_ece:
#         best_T, best_ece = T, e
# print(f'Best T={best_T:.2f} -> ECE={best_ece:.4f}')
print('Temperature-scaling cell — uncomment and run only if 0.10 < ECE <= 0.15.')

## Fallback B — Retrain with active dropout (if ECE > 0.15)

Post-hoc injection means the model never *trained* with dropout. For proper
calibration, rebuild RT-DETR-L with `dropout=0.1` baked into the decoder and
fine-tune. Weight tensor shapes are unchanged by the dropout probability, so the
robust `best.pt` loads cleanly — a short fine-tune (~20-30 epochs) adapts the model.

In [ ]:
# Run ONLY if ECE > 0.15 after temperature scaling.
#
# 1. Copy Ultralytics' rtdetr-l.yaml, edit the RTDETRDecoder head line so the
#    decoder 'dropout' arg is 0.1 (8th positional arg of RTDETRDecoder):
#       [[21,24,27], 1, RTDETRDecoder, [nc, 256, 300, 4, 8, 6, 1024, 0.1]]
# 2. Build the model from the modified yaml and load the robust weights:
#       model_d = RTDETR('rtdetr-l-dropout.yaml')
#       model_d.load(f'{PROJECT_DIR}/models/robust/best.pt')
# 3. Short fine-tune (dropout now active during training):
#       model_d.train(data=LOCAL_YAML, epochs=30, imgsz=640, batch=8,
#                     lr0=5e-5, device=0, project='/content/runs', name='mc_dropout')
# 4. Save to models/robust/best_mcdropout.pt and re-run Steps 2-6.
print('Retrain fallback — see commented steps above. Run only if ECE > 0.15.')

## Phase 5 Completion Checklist

- [ ] MC Dropout verified active (outputs differ across passes)
- [ ] N=30 passes implemented, latency measured
- [ ] ECE <= 0.10 on clean test set
- [ ] VIZ 5.A saved — reliability diagram
- [ ] VIZ 5.B saved — uncertainty overlay (corrupted row redder)
- [ ] Spearman rho > 0.3
- [ ] uncertainty_vs_severity.png saved — >= 3/5 corruptions hit 1.5x ratio
- [ ] `results/mc_dropout_metrics.json` saved

Next → `06_conformal.ipynb`